[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/02_tfidf_vsm_similitud.ipynb)

# Capítulo 2. Ingeniería de características, TF-IDF, VSM y similitud del coseno

Este notebook forma parte del apunte del curso **Analítica Textual** y está preparado para ejecutarse en Google Colab o en Jupyter local.


In [ ]:
# Setup opcional para Colab
# En general, este notebook corre sin instalaciones adicionales.
# Si tu entorno no tiene las librerías base, descomenta la línea siguiente.
# !pip -q install scikit-learn pandas


## Objetivos

En este capítulo trabajaremos sobre la pregunta central de toda analítica textual clásica: ¿cómo convertir documentos en vectores?

Al finalizar deberías poder:

- construir una matriz término-frecuencia;
- explicar el sentido de TF-IDF;
- interpretar el modelo de espacio vectorial;
- calcular similitud del coseno entre documentos.

## 2.1 De texto a números

Los modelos estadísticos no operan directamente sobre frases; necesitan una representación numérica. La opción más simple es la bolsa de palabras, donde cada documento se representa por el conteo de sus términos.

Corpus de ejemplo:


In [ ]:
documentos = [
    "el banco ofrece crédito hipotecario",
    "la fintech mejora la experiencia de pago",
    "el hospital analiza registros clínicos",
    "la clínica mejora la atención de pacientes"
]


## 2.2 Matriz término-frecuencia


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizador = CountVectorizer()
X_tf = vectorizador.fit_transform(documentos)

matriz_tf = pd.DataFrame(
    X_tf.toarray(),
    columns=vectorizador.get_feature_names_out()
)
matriz_tf


Cada fila representa un documento y cada columna una palabra del vocabulario.

## 2.3 TF-IDF

El conteo puro tiene una limitación: palabras frecuentes en casi todos los documentos pesan demasiado. TF-IDF ajusta ese problema:

\[
TFIDF(t, d) = TF(t, d) \times IDF(t)
\]

donde:

- `TF(t, d)` mide la frecuencia del término `t` en el documento `d`;
- `IDF(t)` reduce el peso de términos comunes en el corpus.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(documentos)

matriz_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
).round(3)

matriz_tfidf


## 2.4 Modelo de espacio vectorial

El Vector Space Model representa cada documento como un punto en un espacio de muchas dimensiones. Si el vocabulario tiene 1000 términos, entonces cada documento vive en un espacio de 1000 dimensiones.

La intuición es simple:

- documentos similares quedan cerca;
- documentos distintos quedan lejos;
- consultas pueden representarse del mismo modo que documentos.

## 2.5 Similitud del coseno

La similitud del coseno compara el ángulo entre dos vectores:

\[
\text{coseno}(A, B) = \frac{A \cdot B}{||A|| \, ||B||}
\]

Su valor está entre 0 y 1 cuando usamos vectores no negativos:

- cercano a 1: documentos muy similares;
- cercano a 0: documentos poco relacionados.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similitud = cosine_similarity(X_tfidf)
pd.DataFrame(similitud).round(3)


## 2.6 Recuperación de documentos parecidos

Podemos comparar una consulta con el corpus:


In [ ]:
consulta = ["crédito para vivienda"]
q = tfidf.transform(consulta)

sim_q = cosine_similarity(q, X_tfidf)[0]
resultado = pd.DataFrame({
    "documento": documentos,
    "similitud": sim_q
}).sort_values("similitud", ascending=False)

resultado


Esto ya se parece a un motor de búsqueda elemental.

## 2.7 Decisiones de modelado importantes

`TfidfVectorizer` tiene parámetros muy influyentes:

- `stop_words`: elimina palabras vacías;
- `ngram_range`: incluye secuencias como bigramas;
- `min_df`: descarta términos demasiado raros;
- `max_df`: descarta términos excesivamente comunes.

Ejemplo:


In [ ]:
tfidf_bi = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_bi = tfidf_bi.fit_transform(documentos)
X_bi.shape


## 2.8 Ventajas y límites

Ventajas:

- simple de implementar;
- interpretable;
- muy competitivo en muchos problemas de clasificación.

Límites:

- no captura bien sinonimia ni polisemia;
- genera matrices dispersas y de alta dimensión;
- depende del vocabulario observado.

## Ejercicios

1. Repite el ejemplo usando comentarios de clientes en vez de documentos institucionales.
2. Agrega bigramas al vectorizador y compara el tamaño del vocabulario.
3. Crea una consulta distinta y analiza qué documento recupera primero.

## Idea clave

TF-IDF y el modelo de espacio vectorial siguen siendo herramientas fuertes porque convierten texto en una representación interpretable, robusta y suficiente para muchos problemas reales.
